In [ ]:
"""
Hybrid Energy System Simulation Function
Based on NSGA-II Optimization Algorithm

CORRECTED VERSION with proper unit conversions and cost calculations
"""

import numpy as np
import pandas as pd
from typing import Dict, Tuple, List

import json

import yaml   # pip install pyyaml

def load_system_data(filepath='old_parameters.yaml'):
    """
    Load the old parameter set and config from a YAML file.

    Returns:
        parameters (dict), config (dict)
    """
    with open(filepath, 'r') as f:
        data = yaml.safe_load(f)   # safe_load avoids arbitrary code execution
    print(f"System config loaded from {filepath}")
    return data['parameters'], data['config']


class HybridEnergySystem:
    """
    Simulates a hybrid renewable energy system with:
    - Photovoltaic (PV) panels
    - Wind Turbines (WT)
    - Hydrogen Storage (H2)
    - Fuel Cell (FC)
    - Electrolyzer (EL)
    - Diesel Generator (DG)
    
    UNIT CONVENTIONS:
    - Power: kW (kilowatts)
    - Energy: kWh (kilowatt-hours)
    - Hydrogen mass: kg (kilograms)
    - Time: hours
    - Currency: USD ($)
    """
    
    def __init__(self, parameters: Dict):
        """
        Initialize system parameters.

        Parameters
        ----------
        parameters : dict
            Dictionary containing all system parameters, grouped as follows:

            GENERATOR CONFIGS:
            - rated_PV          : float  - Rated power per PV panel (kW). Default: 0.327
            - v_cut_in          : float  - Wind turbine cut-in speed (m/s). Default: 2.75
            - v_rated           : float  - Wind turbine rated speed (m/s). Default: 9.0
            - rated_power       : float  - Rated power per wind turbine (kW). Default: 25.0
            - Cap_H2            : float  - Capacity of one H2 storage unit (kg). Default: 6
            - Cap_FC            : float  - Rated power of one fuel cell unit (kW). Default: 2
            - Cap_EL            : float  - Rated power of one electrolyzer unit (kW). Default: 2
            - Cap_DG            : float  - Rated power of one diesel generator unit (kW). Default: 50
            - H_min_percentage  : float  - Minimum H2 storage level as fraction of capacity (0-1). Default: 0
            - H_max_percentage  : float  - Maximum H2 storage level override as fraction (0-1). Default: 0

            DIESEL FUEL CURVE CONSTANTS:
            - f_0               : float  - Intercept coefficient (litre/kW/h). Default: 0.246
            - f_1               : float  - Slope coefficient (litre/kWh). Default: 0.08145

            EFFICIENCY PARAMETERS:
            - eta_PV            : float  - PV panel efficiency, 0-1. Default: 0.15
            - eta_FC            : float  - Fuel cell electrical efficiency, 0-1. Default: 0.50
            - eta_EL            : float  - Electrolyzer efficiency, 0-1. Default: 0.70
            - eta_INVT          : float  - Inverter efficiency, 0-1. Default: 0.90
            - H2_LHV            : float  - Hydrogen lower heating value (kWh/kg). Default: 33.3

            CAPITAL COSTS (one-time investment):
            - c_PV              : float  - PV capital cost ($/kW). Default: 1300
            - c_WT              : float  - Wind turbine capital cost ($/kW). Default: 2300
            - c_H2              : float  - Hydrogen storage capital cost ($/kg capacity). Default: 500
            - c_FC_cap          : float  - Fuel cell capital cost ($/kW). Default: 2000
            - c_EL_cap          : float  - Electrolyzer capital cost ($/kW). Default: 1500
            - c_DG_cap          : float  - Diesel generator capital cost ($/kW). Default: 500
            - c_INVT            : float  - Inverter capital cost, flat total ($). Default: 300

            OPERATING COSTS:
            - c_FC              : float  - Fuel cell operating cost ($/kWh produced). Default: 0
            - c_DG              : float  - Diesel generator operating cost ($/kWh produced). Default: 0
            - c_EL              : float  - Electrolyzer operating cost ($/kWh consumed). Default: 0
            - c_DG_FUEL         : float  - Diesel fuel cost ($/litre). Default: 0.82

            OPERATION & MAINTENANCE COSTS:
            - om_PV             : float  - PV O&M ($/kW/year). Default: 10
            - om_WT             : float  - Wind turbine O&M ($/kW/year). Default: 69
            - om_H2             : float  - Hydrogen storage O&M ($/kg/year). Default: 10
            - om_FC             : float  - Fuel cell O&M ($/kW/year). Default: 30
            - om_EL             : float  - Electrolyzer O&M ($/kW/year). Default: 25
            - om_DG             : float  - Diesel generator O&M ($/operating hour). Default: 0.03
            - om_INVT           : float  - Inverter O&M ($). Default: 0

            REPLACEMENT COSTS (present value, per unit capacity):
            - rc_PV             : float  - PV replacement cost ($/kW). Default: 0
            - rc_WT             : float  - Wind turbine replacement cost ($/kW). Default: 1750
            - rc_H2             : float  - Hydrogen storage replacement cost ($/kg). Default: 10
            - rc_FC             : float  - Fuel cell replacement cost ($/kW). Default: 30
            - rc_EL             : float  - Electrolyzer replacement cost ($/kW). Default: 25
            - rc_DG             : float  - Diesel generator replacement cost ($/kW). Default: 500
            - rc_INVT           : float  - Inverter replacement cost, flat total ($). Default: 300

            EMISSION FACTORS:
            - e_FC              : float  - Fuel cell emissions (kg CO2/kWh). Default: 0.0
            - e_DG              : float  - Diesel generator emissions (kg CO2/litre). Default: 2.6391
            - e_EL              : float  - Electrolyzer direct emissions (kg CO2/kWh). Default: 0.0

            ECONOMIC PARAMETERS:
            - T_life            : int    - Project lifetime (years). Default: 20
            - r                 : float  - Annual discount rate, e.g. 0.05 = 5%. Default: 0.05
            - p_grid            : float  - Grid energy selling price ($/kWh). Default: 0.08

            TECHNICAL PARAMETERS:
            - A_PV              : float  - PV panel area per kW capacity (m²/kW). Default: 6.67
            - P_DG_min          : float  - Minimum diesel generator load ratio, 0-1. Default: 0.3

            COMPONENT LIFETIMES (years):
            - life_PV           : int    - PV panel lifetime. Default: 25
            - life_WT           : int    - Wind turbine lifetime. Default: 20
            - life_H2           : int    - Hydrogen storage lifetime. Default: 20
            - life_FC           : int    - Fuel cell lifetime. Default: 10
            - life_EL           : int    - Electrolyzer lifetime. Default: 15
            - life_DG           : int    - Diesel generator lifetime. Default: 15
            - life_INVT         : int    - Inverter lifetime. Default: 15

            FUNCTIONALITY:
            - output_simulation : Bool     - Set True to output .csv file of the simulation
        """
        # =================================================================
        # Hudai Configs
        # =================================================================

        self.load_scaler = parameters.get('load_scaler',1)

        # =================================================================
        # Generator configs
        # =================================================================
        self.rated_PV = parameters.get('rated_PV',0.327)
        self.v_cut_in = parameters.get('v_cut_in', 2.75)   # Cut-in wind speed (m/s)
        self.v_rated =  parameters.get('v_rated', 9.0)  # Rated wind speed (m/s)
        self.rated_power = parameters.get('rated_power', 25.0) #wind turbine rated power
        self.Cap_H2 = parameters.get('Cap_H2',6) #kg Capacity of 1 H2 storage
        self.Cap_FC = parameters.get('Cap_FC',2) #kW Capacity/Rated power of FC
        self.Cap_EL = parameters.get('Cap_EL',2) #kW Capacity/Rated power of Electrolyzer
        self.Cap_DG = parameters.get('Cap_DG',50) #kW Capacity/Rated power of Diesel
        self.H_min_percentage = parameters.get('H_min_percentage',0) 
        self.H_max_percentage = parameters.get('H_max_percentage',0)

        # =================================================================
        # Diesel Constants
        # =================================================================
        self.f_0 = parameters.get('f_0', 0.246)#litre/kW/h
        self.f_1 = parameters.get('f_1', 0.08145)#litre/kWh





        # =================================================================
        # EFFICIENCY PARAMETERS
        # =================================================================
        self.eta_PV = parameters.get('eta_PV', 0.15)  # PV efficiency (fraction)
        self.eta_FC = parameters.get('eta_FC', 0.50)  # Fuel cell efficiency (fraction)
        self.eta_EL = parameters.get('eta_EL', 0.70)  # Electrolyzer efficiency (fraction)
        self.eta_INVT = parameters.get('eta_INVT', 0.90)  # Inverter efficiency (fraction)
        
        # Hydrogen energy content (thermodynamic constant)
        self.H2_LHV = parameters.get('H2_LHV', 33.3)  # kWh/kg (Lower Heating Value)
        
        # =================================================================
        # CAPITAL COSTS ($/unit)
        # =================================================================
        self.c_PV = parameters.get('c_PV', 1300)      # $/kW
        self.c_WT = parameters.get('c_WT', 2300)      # $/kW
        self.c_H2 = parameters.get('c_H2', 500)       # $/kg capacity
        self.c_FC_cap = parameters.get('c_FC_cap', 2000)  # $/kW
        self.c_EL_cap = parameters.get('c_EL_cap', 1500)  # $/kW
        self.c_DG_cap = parameters.get('c_DG_cap', 500)   # $/kW
        self.c_INVT = parameters.get('c_INVT',300)
        
        # =================================================================
        # OPERATING COSTS ($/kWh)
        # =================================================================
        self.c_FC = parameters.get('c_FC', 0)  # $/kWh produced
        self.c_DG = parameters.get('c_DG', 0)  # $/kWh produced (diesel fuel)
        self.c_EL = parameters.get('c_EL', 0)  # $/kWh consumed
        self.c_DG_FUEL = parameters.get('c_DG_FUEL',0.82)  #$/litre diesel consumed

        
        # =================================================================
        # O&M COSTS ($/unit/year)
        # =================================================================
        self.om_PV = parameters.get('om_PV', 10)  # $/kW/year
        self.om_WT = parameters.get('om_WT', 69)  # $/kW/year
        self.om_H2 = parameters.get('om_H2', 10)  # $/kg/year
        self.om_FC = parameters.get('om_FC', 30)  # $/kW/year
        self.om_EL = parameters.get('om_EL', 25)  # $/kW/year
        self.om_DG = parameters.get('om_DG', 0.03)  # $/h
        self.om_INVT = parameters.get('om_INVT',0) 


        # Replacement COSTS(rc) ($/unit/year)
        # =================================================================
        self.rc_PV = parameters.get('rc_PV', 0)  # $/kW
        self.rc_WT = parameters.get('rc_WT', 1750)  # $/kW
        self.rc_H2 = parameters.get('rc_H2', 10)  # $/kg
        self.rc_FC = parameters.get('rc_FC', 30)  # $/kW
        self.rc_EL = parameters.get('rc_EL', 25)  # $/kW
        self.rc_DG = parameters.get('rc_DG', 500)  # $/kW
        self.rc_INVT = parameters.get('rc_INVT',300) # per piece


        
        # =================================================================
        # EMISSION FACTORS (kg CO2/kWh)
        # =================================================================
        self.e_FC = parameters.get('e_FC', 0.0)   # Assuming H2 from renewables
        self.e_DG = parameters.get('e_DG', 2.6391)   # 2.6391 kg/Litre Diesel emissions
        self.e_EL = parameters.get('e_EL', 0.0)   # Electrolyzer direct emissions
        
        # =================================================================
        # ECONOMIC PARAMETERS
        # =================================================================
        self.T_life = parameters.get('T_life', 20)    # Project lifetime (years)
        self.r = parameters.get('r', 0.05)            # Discount rate
        self.p_grid = parameters.get('p_grid', 0.08)  # Grid selling price ($/kWh)
        
        # =================================================================
        # TECHNICAL PARAMETERS
        # =================================================================
        self.A_PV = parameters.get('A_PV', 6.67)      # PV area per kW (m²/kW)
        self.P_DG_min = parameters.get('P_DG_min', 0.3)  # Minimum DG load ratio
        # self.DG_CAPACITY = parameters.get('DG_CAPACITY') #DG capacity
        # self.DG_RATED = parameters.get('DG_RATED',50) #DG Rated Capacity
        
        # =================================================================
        # COMPONENT LIFETIMES (years)
        # =================================================================
        self.life_PV = parameters.get('life_PV', 25)
        self.life_WT = parameters.get('life_WT', 20)
        self.life_H2 = parameters.get('life_H2', 20)
        self.life_FC = parameters.get('life_FC', 10)
        self.life_EL = parameters.get('life_EL', 15)
        self.life_DG = parameters.get('life_DG', 15)
        self.life_INVT = parameters.get('life_INVT', 15)

        # =================================================================
        # FUNCTIONALITY
        # =================================================================
        
        self.output_simulation = parameters.get('output_simulation',False)

    
    
    def wind_power_curve(self, v: float) -> float:
        """
        Calculate wind turbine power output based on wind speed
        Using a simplified power curve model
        
        Parameters
        ----------
        v : float
            Wind speed (m/s)
        rated_power : float
            Rated power of wind turbine (kW)
            Default: 1.0 kW (for per-kW calculations)
            
        Returns
        -------
        float
            Power output (kW)
        """
        # v_cut_in = 2.75   # Cut-in wind speed (m/s)
        # v_rated = 9.0   # Rated wind speed (m/s)
        # # v_cut_out = 25.0 # Cut-out wind speed (m/s)
        # rated_power = 20
        
        if v < self.v_cut_in :
            return 0.0
        elif v >= self.v_cut_in and v < self.v_rated:
            # Cubic relationship between cut-in and rated
            return self.rated_power * ((v**3 - self.v_cut_in**3) / (self.v_rated**3 - self.v_cut_in**3))
        else:  # v_rated <= v <= v_cut_out
            return self.rated_power
    
    
    def calculate_replacement_cost(self, system: Dict, T_life: int, r: float) -> float:
        """
        Calculate the total present value of all component replacement costs
        over the project lifetime.

        A replacement occurs at years: life, 2*life, 3*life, ...
        Any replacement that falls exactly on or after T_life is excluded
        (the project is over; you wouldn't replace something on the last day).

        Parameters
        ----------
        system : dict
            System configuration (N_PV, N_WT, N_H2, N_FC, N_EL, N_DG)
        T_life : int
            Project lifetime (years)
        r : float
            Annual discount rate (e.g. 0.05 = 5%)

        Returns
        -------
        float
            Present value of all replacement costs ($)
        """
        C_rep = 0.0

        # ------------------------------------------------------------------
        # Per-unit components
        # Each tuple: (system_key, capacity_per_unit, replacement_cost_per_unit_capacity, lifetime)
        # Uses rc_* parameters (replacement costs), NOT c_* (capital costs)
        # ------------------------------------------------------------------
        components = [
            ('N_PV', self.rated_PV,    self.rc_PV,  self.life_PV),
            ('N_WT', self.rated_power, self.rc_WT,  self.life_WT),
            ('N_H2', self.Cap_H2,      self.rc_H2,  self.life_H2),
            ('N_FC', self.Cap_FC,      self.rc_FC,  self.life_FC),
            ('N_EL', self.Cap_EL,      self.rc_EL,  self.life_EL),
            ('N_DG', self.Cap_DG,      self.rc_DG,  self.life_DG),
        ]

        for comp_key, capacity_per_unit, rc_per_unit_capacity, life in components:
            n_units = system.get(comp_key, 0)

            # Skip if component not present or outlasts the project
            if n_units == 0 or life <= 0 or life >= T_life:
                continue

            # Total replacement cost for this component (one replacement event)
            total_rc = n_units * capacity_per_unit * rc_per_unit_capacity

            # Replacement years: life, 2*life, ... up to but NOT including T_life
            replacement_year = life
            while replacement_year < T_life:
                C_rep += total_rc / ((1 + r) ** replacement_year)
                replacement_year += life

        # ------------------------------------------------------------------
        # Inverter: single flat cost (c_INVT is a total $ amount, not per-kW)
        # Uses rc_INVT; falls back to c_INVT if rc_INVT is 0
        # ------------------------------------------------------------------
        invt_rc = self.rc_INVT if self.rc_INVT > 0 else self.c_INVT

        if invt_rc > 0 and self.life_INVT > 0 and self.life_INVT < T_life:
            replacement_year = self.life_INVT
            while replacement_year < T_life:
                C_rep += invt_rc / ((1 + r) ** replacement_year)
                replacement_year += self.life_INVT

        return C_rep  
      
    def simulate_year(self, system: Dict, data: pd.DataFrame) -> Tuple[float, float, float, Dict]:
        """
        Simulate one year of system operation
        
        Parameters
        ----------
        system : dict
            System configuration containing:
            - N_PV: number of PV 
            - N_WT: number of Wind turbines 
            - N_H2: Hydrogen storage capacity (kg)
            - N_FC: Fuel cell capacity (kW)
            - N_EL: Electrolyzer capacity (kW)
            - N_DG: Diesel generator capacity (kW)
        
        data : pd.DataFrame
            Hourly data with columns:
            - 'Solar Power','Solar Irradiance (W/sq.m)' or 'Avg Solar Irradiance'
            - 'Wind Speed (m/s)' or 'Avg Wind Speed'
            - 'Community Load' and/or 'RO Load (kWh)'
            
        Returns
        -------
        Tuple[float, float, float, Dict]
            C_total: Total annualized cost ($/year)
            E_total: Total annual emissions (kg CO2/year)
            LPSP: Loss of power supply probability (0-1)
            details: Dictionary with detailed results
        """
        # =================================================================
        # EXTRACT SYSTEM CONFIGURATION
        # =================================================================

        N_PV = system.get('N_PV', 0)
        N_WT = system.get('N_WT', 0)
        Capacity_H2 = system.get('N_H2', 0)*self.Cap_H2
        Capacity_FC = system.get('N_FC', 0)*self.Cap_FC
        Capacity_EL = system.get('N_EL', 0)*self.Cap_EL
        Capacity_DG = system.get('N_DG', 0)*self.Cap_DG
 
        # Validate inputs
        if Capacity_H2 < 0 or Capacity_FC < 0 or Capacity_EL < 0:
            raise ValueError("Capacities cannot be negative")
        
        # =================================================================
        # HYDROGEN STORAGE LIMITS
        # =================================================================
        H_max = Capacity_H2  # Maximum hydrogen storage (kg)
        H_min = self.H_min_percentage * Capacity_H2  # Minimum hydrogen level (10% of capacity)
        
        # =================================================================
        # INITIALIZE TRACKING VARIABLES
        # =================================================================
        C_op = 0.0       # Operating cost ($)
        E_CO2 = 0.0      # CO2 emissions (kg)
        E_unmet = 0.0    # Unmet energy (kWh)
        E_grid = 0.0     # Energy sold to grid (kWh)
        
        E_PV_total = 0.0
        E_WT_total = 0.0
        E_FC_total = 0.0
        E_EL_total = 0.0
        E_DG_total = 0.0

        # Hourly log storage
        hourly_log = []
        
        # Initialize hydrogen storage trajectory
        H = np.zeros(len(data) + 1)
        H[0] = 1 * Capacity_H2  # Start at 100% capacity
        
        # =================================================================
        # DETECT COLUMN NAMES
        # =================================================================
        # if 'Solar Irradiance (W/sq.m)' in data.columns:
        #     irrad_col = 'Solar Irradiance (W/sq.m)'
        # elif 'Avg Solar Irradiance' in data.columns:
        #     irrad_col = 'Avg Solar Irradiance'
        # else:
        #     raise ValueError("Solar irradiance column not found in data")
        

        if 'Solar Power' in data.columns:
            avg_solar_col = 'Solar Power'

        else:
            raise ValueError("Solar Power column not found in data")
        
        if 'Avg Wind Speed NASA (36m)' in data.columns:
            wind_col = 'Avg Wind Speed NASA (36m)'
        elif 'Avg Wind Speed' in data.columns:
            wind_col = 'Avg Wind Speed'
        else:
            raise ValueError("Wind speed column not found in data")
        
        # =================================================================
        # CALCULATE TOTAL LOAD
        # =================================================================
        L = data['Community Load'].values.copy()
        L = self.load_scaler * L
        # if 'RO Load (kWh)' in data.columns:
        #     L = L + 0*data['RO Load (kWh)'].values
        
        L_year = np.sum(L)  # Total annual load (kWh)
        if self.output_simulation:

            print(len(data))          # Should be 8760 for hourly annual data
            print(L.sum())  # Should match ~1,096,946
            print(pd.Series(L).describe())
        
        # =================================================================
        # HOURLY SIMULATION LOOP
        # =================================================================
        for t in range(len(data)):
            # Get hourly inputs
            # I_t = data.iloc[t][irrad_col] / 1000.0  # Convert W/m² to kW/m²
            v_t = data.iloc[t][wind_col]            # Wind speed (m/s)
            PV_t = data.iloc[t][avg_solar_col]
            L_t = L[t]                               # Load (kWh for this hour)

            E_FC = 0.0
            E_DG = 0.0
            E_deficit_logged = 0.0
            E_surplus_logged = 0.0
            E_EL_AC = 0.0
                        
            # ---------------------------------------------------------
            # RENEWABLE ENERGY GENERATION
            # ---------------------------------------------------------
            # PV generation: Power = efficiency × area × irradiance × capacity
            
            E_PV = PV_t * N_PV*self.eta_INVT  # kWh AC
            # Wind generation: Power = power_curve(wind_speed) × capacity
            E_WT = self.wind_power_curve(v_t) * N_WT  # kWh AC
            
            E_RE = E_PV + E_WT  # Total renewable energy AC (kWh)
            
            E_PV_total += E_PV
            E_WT_total += E_WT
            
            # ---------------------------------------------------------
            # NET POWER BALANCE
            # ---------------------------------------------------------
            E_net = E_RE - L_t  # Positive = surplus, Negative = deficit
            
            # =============================================================
            # CASE 1: DEFICIT (E_net < 0)
            # =============================================================
            if E_net < 0:
                E_grid_hour = 0.0
                E_deficit = abs(E_net)  # kWh needed
                E_FC = 0.0
                E_DG = 0.0
                
                # ---------------------------------------------------------
                # STEP 1: TRY FUEL CELL FIRST
                # ---------------------------------------------------------
                if H[t] > H_min and Capacity_FC > 0:
                    # -------------------------------------------------
                    # FUEL CELL OPERATION
                    # DC chain: H2 -> [FC, eta_FC] -> DC -> [Inverter, eta_INVT] -> AC load
                    # -------------------------------------------------
                    # Maximum H2 available for use (kg)
                    H2_available = H[t] - H_min
                    
                    # Maximum energy from available H2 (kWh) - in DC terms (pre-inverter)
                    # Energy = H2_mass × LHV × efficiency
                    E_FC_max_from_H2 = H2_available * self.H2_LHV * self.eta_FC
                    
                    # Maximum energy from FC capacity (kWh in 1 hour) - in DC terms
                    E_FC_max_from_cap = Capacity_FC * 1.0  # kW × 1 hour
                    
                    # Actual FC DC limit is minimum of both constraints
                    E_FC_max = min(E_FC_max_from_H2, E_FC_max_from_cap)
                    
                    # Convert AC deficit to DC equivalent to compare against DC limits
                    E_deficit_DC = E_deficit / self.eta_INVT

                    # FC DC output: limited by DC capacity and DC deficit requirement
                    E_FC_DC = min(E_deficit_DC, E_FC_max)

                    # Convert DC output to AC output delivered to load (through inverter)
                    E_FC = E_FC_DC * self.eta_INVT
                    
                    # H2 consumed (kg) - based on DC energy into inverter
                    # H2_consumed = DC_Energy / (LHV × eta_FC)
                    H2_consumed = E_FC_DC / (self.H2_LHV * self.eta_FC)
                    
                    # Update hydrogen storage---------------------------
                    H[t+1] = max(0, H[t] - H2_consumed)
                    
                    # Add costs and emissions (tracked in AC terms, consistent with load)
                    C_op += self.c_FC * E_FC
                    E_CO2 += self.e_FC * E_FC
                    E_FC_total += E_FC
                    
                    # Update remaining deficit (in AC terms)
                    E_deficit = E_deficit - E_FC
                else:
                    # No fuel cell available, keep current H2 level
                    H[t+1] = H[t]
                
                # ---------------------------------------------------------
                # STEP 2: IF STILL DEFICIT, TRY DIESEL GENERATOR
                # ---------------------------------------------------------
                if E_deficit > 0.001 and Capacity_DG > 0:  # Small tolerance
                    # -------------------------------------------------
                    # DIESEL GENERATOR OPERATION
                    # -------------------------------------------------
                    # Diesel generator must run above minimum load
                    P_DG_min_abs = self.P_DG_min * self.Cap_DG  # Minimum power (kW)
                    # Check if deficit is within operable range
                    if E_deficit >= P_DG_min_abs:
                        # DG can operate
                        E_DG = min(E_deficit, Capacity_DG)  # Limited by capacity
                        #We can choose to run the DG several times to minimize deficit
                        #As long as we are being above the minimum so if deficit is 5c and capacity is c
                        #we run DG 5 times. If deficit is 5c+k, then unmet load is k

                        

                        DG_Litre = self.f_1 * Capacity_DG + self.f_0 * E_DG

                        # Add costs and emissions
                        C_op += self.c_DG_FUEL * DG_Litre
                        E_CO2 += self.e_DG * DG_Litre
                        E_DG_total += E_DG
                        
                        # Update remaining deficit
                        E_deficit = E_deficit - E_DG
                    # else: deficit too small for DG minimum load, remains unmet
                
                # ---------------------------------------------------------
                # STEP 3: ANY REMAINING DEFICIT IS UNMET LOAD
                # ---------------------------------------------------------
                if E_deficit > 0.001:  # Small tolerance
                    E_unmet += E_deficit
            
            # =============================================================
            # CASE 2: SURPLUS (E_net > 0)
            # =============================================================
            elif E_net > 0:
                E_surplus = E_net  # kWh available (AC)
                
                # Check if storage has space and electrolyzer exists
                if H[t] < H_max and Capacity_EL > 0:
                    # -------------------------------------------------
                    # ELECTROLYZER OPERATION
                    # AC chain: AC surplus -> [Rectifier, eta_INVT] -> DC -> [Electrolyzer, eta_EL] -> H2
                    # -------------------------------------------------
                    # Available storage space (kg)
                    H2_space = H_max - H[t]
                    
                    # Maximum energy that can be converted to H2 based on storage space
                    # Energy = H2_mass / (efficiency / LHV)
                    # H2_produced = Energy × efficiency / LHV
                    # So: Energy_max = H2_space × LHV / efficiency  (in DC terms)
                    E_EL_max_from_storage = H2_space * self.H2_LHV / self.eta_EL
                    
                    # Maximum energy from EL capacity (kWh in 1 hour) - in DC terms
                    E_EL_max_from_cap = Capacity_EL * 1.0  # kW × 1 hour

                    # Maximum DC available from AC surplus through rectifier
                    E_surplus_DC = E_surplus * self.eta_INVT

                    # Actual EL DC input is minimum of all constraints
                    E_EL_DC = min(E_surplus_DC, E_EL_max_from_storage, E_EL_max_from_cap)

                    # AC consumed from the bus to supply the rectifier
                    E_EL_AC = E_EL_DC / self.eta_INVT
                    
                    # H2 produced (kg) - based on DC energy into electrolyzer
                    # H2_produced = DC_Energy × efficiency / LHV
                    H2_produced = E_EL_DC * self.eta_EL / self.H2_LHV
                    
                    # Update hydrogen storage
                    H[t+1] = min(H_max, H[t] + H2_produced)
                    
                    # Add costs and emissions (tracked in AC terms, consistent with load)
                    C_op += self.c_EL * E_EL_AC
                    E_CO2 += self.e_EL * E_EL_AC
                    E_EL_total += E_EL_AC
                    
                    # Remaining surplus after electrolysis (both in AC terms)
                    E_leftover = E_surplus - E_EL_AC  # subtract the AC consumed, not DC
                    
                    if E_leftover > 0.001:  # Small tolerance
                        # Sell leftover to grid
                        E_grid += E_leftover
                        E_grid_hour = E_leftover
                        C_op -= self.p_grid * E_leftover  # Revenue (negative cost)
                else:
                    # Storage is full or no electrolyzer, sell all surplus to grid
                    H[t+1] = H[t]
                    E_grid += E_surplus
                    E_grid_hour = E_surplus
                    C_op -= self.p_grid * E_surplus  # Revenue (negative cost)
            
            # =============================================================
            # CASE 3: BALANCED (E_net == 0)
            # =============================================================
            else:
                E_grid_hour = 0.0
                H[t+1] = H[t]
            if self.output_simulation:
                hourly_log.append({
                    'Hour': t,
                    'Community Load': L_t,
                    'Solar Power': PV_t,
                    'Wind Power': self.wind_power_curve(v_t),
                    'PV Generation': E_PV,
                    'WT Generation': E_WT,
                    'FC Generation': E_FC,
                    'DG Generation': E_DG,
                    'Unmet Energy': E_deficit if (E_net < 0 and E_deficit > 0.001) else 0.0,
                    'Surplus Energy': E_grid_hour,
                    'H2 Capacity': H[t+1],
                    })
        
        # Export hourly log to CSV
        if self.output_simulation:

            hourly_df = pd.DataFrame(hourly_log)
            hourly_df.to_csv('simulation_hourly_log.csv', index=False)
            print("Hourly log saved to simulation_hourly_log.csv")

        # =================================================================
        # CALCULATE PERFORMANCE METRICS
        # =================================================================
        # Loss of Power Supply Probability
        
        LPSP = E_unmet / L_year if L_year > 0 else 0.0
        
        # =================================================================
        # CALCULATE COSTS
        # =================================================================
        # Capital cost (present value)
        C_cap = (self.c_PV * N_PV*self.rated_PV + 
                 self.c_WT * N_WT*self.rated_power + #wind 
                 self.c_H2 * Capacity_H2 + 
                 self.c_FC_cap * Capacity_FC + 
                 self.c_EL_cap * Capacity_EL + 
                 self.c_DG_cap * Capacity_DG +
                 self.c_INVT)
        
        # Annual O&M cost
        # print(f"CapacityDG = {Capacity_DG}")
        C_om_annual = (self.om_PV * N_PV*self.rated_PV + 
                       self.om_WT * N_WT*self.rated_power + 
                       self.om_H2 * Capacity_H2 + 
                       self.om_FC * Capacity_FC + 
                       self.om_EL * Capacity_EL + 
                       self.om_DG * (E_DG_total/Capacity_DG if Capacity_DG>0 else 0))
        
        # Replacement cost (present value)
        C_rep = self.calculate_replacement_cost(system, self.T_life, self.r)
        
        # Capital Recovery Factor (CRF)
        # Converts present value to equivalent annual cost
        if self.r > 0:
            CRF = (self.r * (1 + self.r) ** self.T_life) / ((1 + self.r) ** self.T_life - 1)
        else:
            CRF = 1.0 / self.T_life
        
        # Total annualized cost
        # = Annualized capital + Annualized replacement + Annual O&M + Annual operating
        C_total = (C_cap + C_rep) * CRF + C_om_annual + C_op
        LCOE = C_total/L_year
        
        # Total emissions
        E_total = E_CO2
        
        # =================================================================
        # PREPARE DETAILED RESULTS
        # =================================================================
        details = {
            'C_cap': C_cap,
            'C_rep': C_rep,
            'C_om_annual': C_om_annual,
            'C_op': C_op,
            'CRF': CRF,
            'E_PV_total': E_PV_total,
            'E_WT_total': E_WT_total,
            'E_FC_total': E_FC_total,
            'E_EL_total': E_EL_total,
            'E_DG_total': E_DG_total,
            'E_grid': E_grid,
            'E_unmet': E_unmet,
            'L_year': L_year,
            'H_trajectory': H,
            'hourly_df' : hourly_df if self.output_simulation else None,
            'LCOE':LCOE
        }
        if self.output_simulation:

            open(f"{C_total/details['L_year']:.2f}_{LPSP:.2f}_N_PV_{system.get('N_PV',0)}_N_WT_{system.get('N_WT',0)}_N_H2_{system.get('N_H2',0)}_N_FC_{system.get('N_FC',0)}_N_EL_{system.get('N_EL',0)}_N_DG_{system.get('N_DG',0)}.txt", 'w').write(f"LPSP,LCOE,N_PV,N_WT,N_H2,N_FC,N_EL,N_DG\n{LPSP},{C_total/details['L_year']:.6f},{system.get('N_PV',0)},{system.get('N_WT',0)},{system.get('N_H2',0)},{system.get('N_FC',0)},{system.get('N_EL',0)},{system.get('N_DG',0)}")
        
        return C_total, E_total, LPSP, details

In [2]:
"""
Example Script: Running the CORRECTED Hybrid Energy System Simulation

This script demonstrates the corrected version with proper unit conversions
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



def run_single_simulation(data,parameters,config):

    """


    Run a single simulation with a given system configuration


    """
    print("="*80)
    print("CORRECTED HYBRID ENERGY SYSTEM SIMULATION")
    print("="*80)

    # =========================================================================
    # CREATE SYSTEM INSTANCE
    # =========================================================================
    system = HybridEnergySystem(parameters)
    


    # Derived capacities for display
    cap_PV  = config['N_PV']  * system.rated_PV     # kW
    cap_WT  = config['N_WT']  * system.rated_power  # kW
    cap_H2  = config['N_H2']  * system.Cap_H2       # kg
    cap_FC  = config['N_FC']  * system.Cap_FC        # kW
    cap_EL  = config['N_EL']  * system.Cap_EL        # kW
    cap_DG  = config['N_DG']  * system.Cap_DG        # kW
    
    print("\n" + "="*80)
    print("SYSTEM CONFIGURATION")
    print("="*80)
    print(f"  PV Capacity: RatedPV {system.rated_PV:>4}kW              {config['N_PV']:>4} panels  ->  {cap_PV:>8.2f} kW")
    print(f"  Wind Turbine Capacity: Rated Power {system.rated_power:>4}kW    {config['N_WT']:>4} turbines ->  {cap_WT:>8.2f} kW")
    print(f"  H2 Storage Capacity: CapH2 {system.Cap_H2:>4}kg      {config['N_H2']:>4} units   ->  {cap_H2:>8.2f} kg")
    print(f"  Fuel Cell Capacity: CapFC {system.Cap_FC:>4}kW       {config['N_FC']:>4} units   ->  {cap_FC:>8.2f} kW")
    print(f"  Electrolyzer Capacity: CapEL {system.Cap_EL:>4}kW    {config['N_EL']:>4} units   ->  {cap_EL:>8.2f} kW")
    print(f"  Diesel Generator: CapDG {system.Cap_DG:>4}kW         {config['N_DG']:>4} units   ->  {cap_DG:>8.2f} kW")
    
    print("\n" + "="*80)
    print("EFFICIENCY DETAILS")
    print("="*80)
    print(f"  PV Efficiency:            {parameters['eta_PV']*100:>6.1f} %")
    print(f"  Fuel Cell Efficiency:     {parameters['eta_FC']*100:>6.1f} %")
    print(f"  Electrolyzer Efficiency:  {parameters['eta_EL']*100:>6.1f} %")
    print(f"  Inverter Efficiency:      {parameters['eta_INVT']*100:>6.1f} %")
    print(f"  H2 Lower Heating Value:   {parameters['H2_LHV']:>6.1f} kWh/kg")
    print()
    print(f"  FC Output:                {parameters['eta_FC']*parameters['H2_LHV']:>6.2f} kWh per kg H2")
    print(f"  EL Input Required:        {parameters['H2_LHV']/parameters['eta_EL']:>6.2f} kWh per kg H2")
    print(f"  Round-trip Efficiency:    {parameters['eta_FC']*parameters['eta_EL']*100:>6.1f} %")
    
    
    print(f"  Data Points:              {len(data):>8} hours")
    print(f"  Represents:               {len(data)/24:>8.1f} days")
    
    # =========================================================================
    # RUN SIMULATION
    # =========================================================================
    print("\n" + "="*80)
    print("RUNNING SIMULATION...")
    print("="*80)
    
    C_total, E_total, LPSP, details = system.simulate_year(config, data)
    
    # =========================================================================
    # DISPLAY RESULTS
    # =========================================================================
    print("\n" + "="*80)
    print("SIMULATION RESULTS")
    print("="*80)
    
    print("\n--- ECONOMIC PERFORMANCE ---")
    print(f"  Total Annualized Cost:    ${C_total:>12,.2f} /year")
    print(f"  Capital Cost:             ${details['C_cap']:>12,.2f}")
    print(f"  Replacement Cost:         ${details['C_rep']:>12,.2f}")
    print(f"  Annual O&M Cost:          ${details['C_om_annual']:>12,.2f} /year")
    print(f"  Annual Operating Cost:    ${details['C_op']:>12,.2f} /year")
    print(f"  Capital Recovery Factor:  {details['CRF']:>12.6f}")
    
    print("\n--- ENVIRONMENTAL PERFORMANCE ---")
    print(f"  Total Annual Emissions:   {E_total:>12,.2f} kg CO2/year")
    print(f"  Emission Intensity:       {E_total/details['L_year']*1000:>12.4f} g CO2/kWh")
    
    print("\n--- RELIABILITY PERFORMANCE ---")
    print(f"  Loss of Power Supply:     {LPSP*100:>12.4f} %")
    print(f"  Reliability:              {(1-LPSP)*100:>12.4f} %")
    print(f"  Unmet Energy:             {details['E_unmet']:>12,.2f} kWh/year")

    # -------------------------------------------------------------------------
    # Pre-compute derived quantities for energy breakdown
    # -------------------------------------------------------------------------
    E_PV        = details['E_PV_total']
    E_WT        = details['E_WT_total']
    E_FC        = details['E_FC_total']
    E_DG        = details['E_DG_total']
    E_EL        = details['E_EL_total']
    E_spilled   = details['E_grid']
    E_unmet     = details['E_unmet']
    L_year      = details['L_year']

    E_RE_generated  = E_PV + E_WT                        # total renewable generated
    E_total_gen     = E_RE_generated + E_FC + E_DG       # everything generated

    # Renewable energy that actually served the load (not spilled)
    E_RE_consumed   = max(0.0, E_RE_generated - E_spilled)

    # Total energy actually consumed by the load
    E_consumed      = L_year - E_unmet                   # = served load

    # Non-renewable share consumed
    E_nonRE_consumed = E_consumed - E_RE_consumed        # FC + DG portion

    # Fractions of CONSUMED energy
    frac_RE_consumed    = E_RE_consumed    / E_consumed  * 100 if E_consumed > 0 else 0
    frac_nonRE_consumed = E_nonRE_consumed / E_consumed  * 100 if E_consumed > 0 else 0

    # Fractions of TOTAL GENERATION
    frac_PV_gen  = E_PV / E_total_gen * 100 if E_total_gen > 0 else 0
    frac_WT_gen  = E_WT / E_total_gen * 100 if E_total_gen > 0 else 0
    frac_FC_gen  = E_FC / E_total_gen * 100 if E_total_gen > 0 else 0
    frac_DG_gen  = E_DG / E_total_gen * 100 if E_total_gen > 0 else 0
    frac_RE_gen  = E_RE_generated / E_total_gen * 100 if E_total_gen > 0 else 0

    print("\n--- GENERATION (What Was Produced) ---")
    print(f"  Total Generation:         {E_total_gen:>12,.2f} kWh/year  (100.0%)")
    print(f"  - Renewable Subtotal:    {E_RE_generated:>12,.2f} kWh/year  ({frac_RE_gen:>5.1f}%)")
    print(f"      - PV:                {E_PV:>12,.2f} kWh/year  ({frac_PV_gen:>5.1f}%)")
    print(f"      - Wind:              {E_WT:>12,.2f} kWh/year  ({frac_WT_gen:>5.1f}%)")
    print(f"  - Fuel Cell:             {E_FC:>12,.2f} kWh/year  ({frac_FC_gen:>5.1f}%)")
    print(f"  - Diesel Generator:      {E_DG:>12,.2f} kWh/year  ({frac_DG_gen:>5.1f}%)")

    print("\n--- GENERATION ROUTING (Where Did It Go) ---")
    print(f"  Total Generated:          {E_total_gen:>12,.2f} kWh/year")
    print(f"  - Served Load:           {E_consumed:>12,.2f} kWh/year  ({E_consumed/E_total_gen*100:>5.1f}%)")
    print(f"  - Spilled to Grid:       {E_spilled:>12,.2f} kWh/year  ({E_spilled/E_total_gen*100:>5.1f}%)")
    print(f"  - Electrolyzer Input:    {E_EL:>12,.2f} kWh/year  ({E_EL/E_total_gen*100:>5.1f}%)")

    print("\n--- LOAD BREAKDOWN (What the Load Actually Received) ---")
    print(f"  Total Load Demand:        {L_year:>12,.2f} kWh/year  (100.0%)")
    print(f"  - Served:                {E_consumed:>12,.2f} kWh/year  ({E_consumed/L_year*100:>5.1f}%)")
    print(f"      - From Renewables:   {E_RE_consumed:>12,.2f} kWh/year  ({frac_RE_consumed:>5.1f}% of served)")
    print(f"      - From Non-RE:       {E_nonRE_consumed:>12,.2f} kWh/year  ({frac_nonRE_consumed:>5.1f}% of served)")
    print(f"  - Unmet:                 {E_unmet:>12,.2f} kWh/year  ({E_unmet/L_year*100:>5.1f}%)")

    print("\n--- RENEWABLE FRACTION (Three Ways to Measure) ---")
    print(f"  RE Generated / Load:      {E_RE_generated/L_year*100:>12.2f} %  <- inflated (counts spilled RE)")
    print(f"  RE Generated / Total Gen: {frac_RE_gen:>12.2f} %  <- pie chart number")
    print(f"  RE Consumed / Load Served:{frac_RE_consumed:>12.2f} %  <- most honest")

    print("\n--- COST & PERFORMANCE SUMMARY ---")
    LCOE = C_total / L_year
    print(f"  Levelized Cost (LCOE):    ${LCOE:>12.4f} /kWh")

    # Capital cost breakdown
    c_pv_cost  = parameters['c_PV']     * cap_PV
    c_wt_cost  = parameters['c_WT']     * cap_WT
    c_h2_cost  = parameters['c_H2']     * cap_H2
    c_fc_cost  = parameters['c_FC_cap'] * cap_FC
    c_el_cost  = parameters['c_EL_cap'] * cap_EL
    c_dg_cost  = parameters['c_DG_cap'] * cap_DG

    print("\n--- CAPITAL COST BREAKDOWN ---")
    print(f"  Total Capital Cost:       ${details['C_cap']:>12,.2f}  (100.0%)")
    print(f"  - PV System:             ${c_pv_cost:>12,.2f}  ({c_pv_cost/details['C_cap']*100:>5.1f}%)")
    print(f"  - Wind Turbines:         ${c_wt_cost:>12,.2f}  ({c_wt_cost/details['C_cap']*100:>5.1f}%)")
    print(f"  - H2 Storage:            ${c_h2_cost:>12,.2f}  ({c_h2_cost/details['C_cap']*100:>5.1f}%)")
    print(f"  - Fuel Cell:             ${c_fc_cost:>12,.2f}  ({c_fc_cost/details['C_cap']*100:>5.1f}%)")
    print(f"  - Electrolyzer:          ${c_el_cost:>12,.2f}  ({c_el_cost/details['C_cap']*100:>5.1f}%)")
    print(f"  - Diesel Generator:      ${c_dg_cost:>12,.2f}  ({c_dg_cost/details['C_cap']*100:>5.1f}%)")

    print("\n" + "="*80)

    return C_total, E_total, LPSP, system, config, data, details

def plot_results(details, config, system,C_total,LPSP):
    """
    Create visualization of system performance.

    Parameters
    ----------
    details : dict
        Detailed results dict returned by simulate_year()
    config : dict
        System configuration (N_PV, N_WT, N_H2, N_FC, N_EL, N_DG)
    system : HybridEnergySystem
        Instantiated system object (needed for capacities like Cap_H2)
    """
    H_min_level = system.H_min_percentage * config['N_H2'] * system.Cap_H2
    H_max_level = config['N_H2'] * system.Cap_H2

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # --- Hydrogen storage trajectory ---
    hours = np.arange(len(details['H_trajectory']))
    axes[0].plot(hours / 24, details['H_trajectory'], linewidth=0.8, label='H2 Level')
    axes[0].axhline(y=H_max_level,  color='red',    linestyle='--', linewidth=1.0, label=f'Max Capacity ({H_max_level:.0f} kg)')
    axes[0].axhline(y=H_min_level,  color='orange', linestyle='--', linewidth=1.0, label=f'Min Level ({H_min_level:.0f} kg)')
    axes[0].set_xlabel('Day')
    axes[0].set_ylabel('Hydrogen Storage (kg)')
    axes[0].set_title('Hydrogen Storage Level Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # --- Energy generation mix pie chart ---
    energy_sources = ['PV', 'Wind', 'Fuel Cell', 'Diesel']
    energy_values  = [
        details['E_PV_total'],
        details['E_WT_total'],
        details['E_FC_total'],
        details['E_DG_total'],
    ]
    
    # Filter out zero/negligible values for a cleaner chart
    filtered = [(s, v) for s, v in zip(energy_sources, energy_values) if v > 0]
    
    if filtered:
        labels, values = zip(*filtered)
        axes[1].pie(values, labels=labels, autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Energy Generation Mix')
    else:
        axes[1].text(0.5, 0.5, 'No Energy Generated', ha='center', va='center',
                     transform=axes[1].transAxes)
        axes[1].set_title('Energy Generation Mix')
    
    plt.tight_layout()
    png_path = f'simulation_results__{C_total/details['L_year']}_{LPSP}_cutin2.75_{config['N_PV']}_{config['N_WT']}_{config['N_FC']}_{config['N_H2']}_{config['N_EL']}_{config['N_DG']}.png'
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    print("\nPlot saved to: simulation_results_cutin2.75.png")
    plt.close()




file_path = 'semi_final_load.xlsx'


# =========================================================================
# LOAD DATA
# =========================================================================
try:
    data = pd.read_excel(file_path)
    scaling_factor = 1.50# Change this to any value > 1 to scale up, < 1 to scale down
    data['Community Load'] = data['Community Load'] * scaling_factor
    print("\n" + "="*80)
    print("DATA LOADED")
    print("="*80)
    print(f"  Source: {file_path}")
except FileNotFoundError:
    try:
        print("Trying Alternative combined")
        data = pd.read_excel('data/combined.xlsx')
        print("\n" + "="*80)
        print("DATA LOADED")
        print("="*80)
        print(f"  Source: data/combined.xlsx")
    except FileNotFoundError:
        print("\n" + "="*80)
        print("ERROR: Could not load data file!")
        print("="*80)
        print("Please ensure one of the following exists:")
        print(f"  - {file_path}")
        print("  - data/combined.xlsx")


    
# =========================================================================
# DEFINE CORRECTED SYSTEM PARAMETERS
# =========================================================================

# parameters = {
# # =================================================================
# # GENERATOR CONFIGS
# # =================================================================
# 'rated_PV': 0.327,          # kW - rated power per PV panel
# 'v_cut_in': 2.75,           # m/s - cut-in wind speed
# 'v_rated': 9.0,             # m/s - rated wind speed
# 'rated_power': 25.0,        # kW - wind turbine rated power
# 'Cap_H2': 6,                # kg - capacity of 1 H2 storage unit
# 'Cap_FC': 2,                # kW - rated power per fuel cell unit
# 'Cap_EL': 2,                # kW - rated power per electrolyzer unit
# 'Cap_DG': 50,               # kW - rated power per diesel generator unit
# 'H_min_percentage': 0,      # fraction - minimum H2 storage level (0 = 0%)
# 'H_max_percentage': 0,      # fraction - maximum H2 storage level override

# # =================================================================
# # DIESEL CONSTANTS
# # =================================================================
# 'f_0': 0.246,               # litre/kW/h - diesel curve intercept coefficient
# 'f_1': 0.08145,             # litre/kWh  - diesel curve slope coefficient

# # =================================================================
# # EFFICIENCY PARAMETERS
# # =================================================================
# 'eta_PV': 0.15,             # fraction - PV panel efficiency (15%)
# 'eta_FC': 0.50,             # fraction - fuel cell efficiency (50%)
# 'eta_EL': 0.70,             # fraction - electrolyzer efficiency (70%)
# 'eta_INVT': 0.90,           # fraction - inverter efficiency (90%)
# 'H2_LHV': 33.3,             # kWh/kg   - hydrogen lower heating value

# # =================================================================
# # CAPITAL COSTS
# # =================================================================
# 'c_PV': 1500,               # $/kW     - PV capital cost
# 'c_WT': 3000,               # $/kW     - wind turbine capital cost
# 'c_H2': 500,                # $/kg     - hydrogen storage capital cost
# 'c_FC_cap': 2000,           # $/kW     - fuel cell capital cost
# 'c_EL_cap': 1500,           # $/kW     - electrolyzer capital cost
# 'c_DG_cap': 400,            # $/kW     - diesel generator capital cost
# 'c_INVT': 300,              # $        - inverter capital cost (flat, not per-kW)

# # =================================================================
# # OPERATING COSTS
# # =================================================================
# 'c_FC': 0,                  # $/kWh    - fuel cell operating cost per kWh produced
# 'c_DG': 0,                  # $/kWh    - diesel operating cost per kWh produced
# 'c_EL': 0,                  # $/kWh    - electrolyzer operating cost per kWh consumed
# 'c_DG_FUEL': 0.82,          # $/litre  - diesel fuel cost

# # =================================================================
# # O&M COSTS
# # =================================================================
# 'om_PV': 20,                # $/kW/year  - PV O&M
# 'om_WT': 50,                # $/kW/year  - wind turbine O&M
# 'om_H2': 10,                # $/kg/year  - hydrogen storage O&M
# 'om_FC': 30,                # $/kW/year  - fuel cell O&M
# 'om_EL': 25,                # $/kW/year  - electrolyzer O&M
# 'om_DG': 0.03,              # $/h        - diesel generator O&M (per operating hour)
# 'om_INVT': 0,               # $          - inverter O&M

# # =================================================================
# # REPLACEMENT COSTS
# # =================================================================
# 'rc_PV': 0,                 # $/kW  - PV replacement cost
# 'rc_WT': 1750,              # $/kW  - wind turbine replacement cost
# 'rc_H2': 10,                # $/kg  - hydrogen storage replacement cost
# 'rc_FC': 30,                # $/kW  - fuel cell replacement cost
# 'rc_EL': 25,                # $/kW  - electrolyzer replacement cost
# 'rc_DG': 500,               # $/kW  - diesel generator replacement cost
# 'rc_INVT': 300,             # $     - inverter replacement cost (flat, per unit)

# # =================================================================
# # EMISSION FACTORS
# # =================================================================
# 'e_FC': 0.0,                # kg CO2/kWh    - fuel cell emissions (green H2 = 0)
# 'e_DG': 2.6391,             # kg CO2/litre  - diesel generator emissions
# 'e_EL': 0.0,                # kg CO2/kWh    - electrolyzer direct emissions

# # =================================================================
# # ECONOMIC PARAMETERS
# # =================================================================
# 'T_life': 20,               # years    - project lifetime
# 'r': 0.05,                  # fraction - annual discount rate (5%)
# 'p_grid': 0.08,             # $/kWh    - grid energy selling price

# # =================================================================
# # TECHNICAL PARAMETERS
# # =================================================================
# 'A_PV': 6.67,               # m²/kW   - PV area per kW capacity
# 'P_DG_min': 0.3,            # fraction - minimum diesel generator load ratio (30%)

# # =================================================================
# # COMPONENT LIFETIMES
# # =================================================================
# 'life_PV': 25,              # years - PV panel lifetime
# 'life_WT': 20,              # years - wind turbine lifetime
# 'life_H2': 20,              # years - hydrogen storage lifetime
# 'life_FC': 10,              # years - fuel cell lifetime
# 'life_EL': 15,              # years - electrolyzer lifetime
# 'life_DG': 15,              # years - diesel generator lifetime
# 'life_INVT': 15,            # years - inverter lifetime

# 'output_simulation' : True
# }
    

# # =========================================================================
# # DEFINE SYSTEM CONFIGURATION
# # =========================================================================

# config = {
#     'N_PV': 2000,      # number of PV panels
#     'N_WT': 17,       # number of wind turbines
#     'N_H2': 0,      # number of H2 storage units
#     'N_FC': 0,       # number of fuel cell units
#     'N_EL': 0,       # number of electrolyzer units
#     'N_DG': 2,       # number of diesel generator units
# }
#-------------------------------------------------------------OLD ONE--------------------------------




parameters,config = load_system_data(filepath="MAYBE2.yaml")
# Run single simulation
result = run_single_simulation(data,parameters,config)

if result is not None:
    C_total, E_total, LPSP, system, config, data, details = result
    
    # Create plots — pass system so plot_results can access capacities
    plot_results(details, config, system,C_total,LPSP)
    
    # Uncomment to run sensitivity analysis
    # sensitivity_results = sensitivity_analysis()


DATA LOADED
  Source: semi_final_load.xlsx
System config loaded from MAYBE2.yaml
CORRECTED HYBRID ENERGY SYSTEM SIMULATION

SYSTEM CONFIGURATION
  PV Capacity: RatedPV 0.327kW              2821 panels  ->    922.47 kW
  Wind Turbine Capacity: Rated Power  5.0kW      60 turbines ->    300.00 kW
  H2 Storage Capacity: CapH2    1kg        72 units   ->     72.00 kg
  Fuel Cell Capacity: CapFC    1kW        116 units   ->    116.00 kW
  Electrolyzer Capacity: CapEL    1kW      68 units   ->     68.00 kW
  Diesel Generator: CapDG    3kW           43 units   ->    129.00 kW

EFFICIENCY DETAILS
  PV Efficiency:              21.4 %
  Fuel Cell Efficiency:       60.0 %
  Electrolyzer Efficiency:    75.0 %
  Inverter Efficiency:        96.0 %
  H2 Lower Heating Value:     39.7 kWh/kg

  FC Output:                 23.83 kWh per kg H2
  EL Input Required:         52.96 kWh per kg H2
  Round-trip Efficiency:      45.0 %
  Data Points:                  8760 hours
  Represents:                  365.